In [0]:
from pyspark.sql import functions as F

target_table = "customer360_dev.silver.customers"

silver_df = spark.table(target_table)

incremental_bronze_df = (
    spark.table(
        "customer360_dev.bronze.customers"
    )
    .filter(
        F.col("_load_type") == "INCREMENTAL"
    )
)

customer_cdc_df = spark.table(
    "customer360_dev.control.customer_cdc_work"
)

In [0]:
run_id_row = (
    customer_cdc_df
    .select("_pipeline_run_id")
    .filter(
        F.col("_pipeline_run_id").isNotNull()
    )
    .first()
)

run_id = (
    run_id_row["_pipeline_run_id"]
    if run_id_row
    else None
)

print("Reconciling Run ID:", run_id)

Reconciling Run ID: None


In [0]:
insert_count = (
    customer_cdc_df
    .filter(F.col("cdc_action") == "INSERT")
    .count()
)

update_count = (
    customer_cdc_df
    .filter(F.col("cdc_action") == "UPDATE")
    .count()
)

unchanged_count = (
    customer_cdc_df
    .filter(F.col("cdc_action") == "UNCHANGED")
    .count()
)

classified_count = (
    insert_count
    + update_count
    + unchanged_count
)

print(f"INSERT     : {insert_count:,}")
print(f"UPDATE     : {update_count:,}")
print(f"UNCHANGED  : {unchanged_count:,}")
print(f"CLASSIFIED : {classified_count:,}")

INSERT     : 0
UPDATE     : 0
UNCHANGED  : 0
CLASSIFIED : 0


In [0]:
source_distinct_customers = (
    incremental_bronze_df
    .select("customer_id")
    .distinct()
    .count()
)

classification_difference = (
    source_distinct_customers
    - classified_count
)

print(
    f"Distinct incremental customers : "
    f"{source_distinct_customers:,}"
)

print(
    f"Classified customers           : "
    f"{classified_count:,}"
)

print(
    f"Difference                     : "
    f"{classification_difference:,}"
)

Distinct incremental customers : 0
Classified customers           : 0
Difference                     : 0


In [0]:
insert_missing_count = (
    customer_cdc_df
    .filter(F.col("cdc_action") == "INSERT")
    .select("customer_id")
    .join(
        silver_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "INSERT customers missing from Silver:",
    insert_missing_count
)

INSERT customers missing from Silver: 0


In [0]:
update_missing_count = (
    customer_cdc_df
    .filter(F.col("cdc_action") == "UPDATE")
    .select("customer_id")
    .join(
        silver_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "UPDATE customers missing from Silver:",
    update_missing_count
)

UPDATE customers missing from Silver: 0


In [0]:
duplicate_customer_ids = (
    silver_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate customer IDs in Silver:",
    duplicate_customer_ids
)

Duplicate customer IDs in Silver: 0


In [0]:
incremental_rows_in_silver = (
    silver_df
    .filter(
        F.col("_load_type") == "INCREMENTAL"
    )
    .count()
)

print(
    "Silver rows currently marked INCREMENTAL:",
    incremental_rows_in_silver
)

Silver rows currently marked INCREMENTAL: 0


In [0]:
reconciliation_pass = (
    classification_difference == 0
    and insert_missing_count == 0
    and update_missing_count == 0
    and duplicate_customer_ids == 0
)

status = (
    "PASS"
    if reconciliation_pass
    else "FAIL"
)

print("Customer CDC Reconciliation:", status)

Customer CDC Reconciliation: PASS


In [0]:
result = [
    {
        "entity": "customers",
        "source_distinct_customers": source_distinct_customers,
        "insert_count": insert_count,
        "update_count": update_count,
        "unchanged_count": unchanged_count,
        "classified_count": classified_count,
        "classification_difference": classification_difference,
        "insert_missing_in_silver": insert_missing_count,
        "update_missing_in_silver": update_missing_count,
        "duplicate_customer_ids": duplicate_customer_ids,
        "status": status
    }
]

display(
    spark.createDataFrame(result)
)

classification_difference,classified_count,duplicate_customer_ids,entity,insert_count,insert_missing_in_silver,source_distinct_customers,status,unchanged_count,update_count,update_missing_in_silver
0,0,0,customers,0,0,0,PASS,0,0,0
